# Day 4 · Notebook 4 — Controlling Agent Behavior

Three controls, in the order they will fight you:

1. **Loop** — stop the runaway (spin-detection + a `handoff_to_human` fallback tool)
2. **Inference** — tune sampling (and know when *not* to, including reasoning models)
3. **Hallucination** — five layers, ending in a real contextual-grounding **guardrail**

This notebook is standalone: it re-defines the compact tools + loop from notebook 2, then improves them. **us-east-1**; model calls need credentials + Claude access.

In [ ]:
# Colab: uncomment.  VS Code venv: skip.
# !pip install -q boto3
import boto3, botocore, json

REGION       = "us-east-1"
CLAUDE_MODEL = "us.anthropic.claude-3-5-haiku-20241022-v1:0"   # cheap workhorse (us. profile)
CLAUDE_37    = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"  # supports manual extended thinking

bedrock         = boto3.client("bedrock",         region_name=REGION)  # CONTROL: guardrails
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)  # INFERENCE: converse, apply_guardrail

# --- tools + system prompt from notebook 2 (compact) ---
BOOKINGS = {"ABC123": {"pnr": "ABC123", "flight": "6E-203", "origin": "BLR", "dest": "DEL",
                       "status": "DISRUPTED", "disruption": "fog at DEL"}}
def lookup_booking(pnr):
    b = BOOKINGS.get((pnr or "").upper()); return {"found": bool(b), **(b or {"pnr": pnr})}
def get_disruption_reason(pnr):
    b = BOOKINGS.get((pnr or "").upper()); return {"pnr": pnr, "reason": (b or {}).get("disruption", "unknown")}
def get_rebooking_options(pnr):
    return {"pnr": (pnr or "").upper(), "options": [{"flight": "6E-415", "dep": "18:40"}, {"flight": "6E-422", "dep": "21:10"}]}

TOOL_IMPL = {"lookup_booking": lookup_booking, "get_disruption_reason": get_disruption_reason,
             "get_rebooking_options": get_rebooking_options}

def _spec(name, desc, props, req):
    return {"toolSpec": {"name": name, "description": desc,
            "inputSchema": {"json": {"type": "object", "properties": props, "required": req}}}}
TOOLS = [
    _spec("lookup_booking", "Look up a booking by PNR.", {"pnr": {"type": "string"}}, ["pnr"]),
    _spec("get_disruption_reason", "Why a flight was disrupted, by PNR.", {"pnr": {"type": "string"}}, ["pnr"]),
    _spec("get_rebooking_options", "Alternative flights for a PNR.", {"pnr": {"type": "string"}}, ["pnr"]),
]
SYSTEM = [{"text": "You are TravelMind. Use ONLY tool facts. Never invent a PNR, flight, or time. "
                   "If you cannot fulfil a request with the tools, use handoff_to_human. Under 80 words."}]

def converse_text(model_id, prompt, system=None, temperature=0.0, max_tokens=300):
    kw = {"modelId": model_id, "messages": [{"role": "user", "content": [{"text": prompt}]}],
          "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system: kw["system"] = [{"text": system}]
    return bedrock_runtime.converse(**kw)["output"]["message"]["content"][0]["text"]

print("base tools + helpers ready")

## 1. Loop control

In notebook 2 the loop ran away on "confirm and pay" because there was **no booking tool and no fallback**. Three fixes, applied together:

- **Fix 1 — `max_turns`** (already in the loop): a hard ceiling. Non-negotiable.
- **Fix 2 — spin-detection**: if the model makes the *same call with the same input* twice, it is not progressing — break.
- **Fix 3 — fallback tool**: add `handoff_to_human(reason)` so the model always has a **legal move**. The loop ends because it *succeeds at handing off*, not because a timer fired.

In [ ]:
def handoff_to_human(reason):
    return {"status": "ESCALATED", "ticket": f"T-{abs(hash(reason)) % 10000:04d}", "reason": reason}
TOOL_IMPL["handoff_to_human"] = handoff_to_human

TOOL_CONFIG = {"tools": TOOLS + [
    _spec("handoff_to_human",
          "Use when you cannot fulfil the request with the other tools "
          "(e.g. the customer wants to confirm, cancel, or pay for a booking). Hands off to a human agent.",
          {"reason": {"type": "string"}}, ["reason"]),
]}

def run_agent_v2(user_text, max_turns=8, temp_schedule=None, verbose=True):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    last_sigs = None
    for turn in range(1, max_turns + 1):
        temp = temp_schedule(turn) if temp_schedule else 0.0
        resp = bedrock_runtime.converse(modelId=CLAUDE_MODEL, messages=messages, system=SYSTEM,
            toolConfig=TOOL_CONFIG, inferenceConfig={"temperature": temp, "maxTokens": 400})
        out = resp["output"]["message"]; messages.append(out); stop = resp["stopReason"]
        if verbose: print(f"[turn {turn}] temp={temp:.1f} stopReason={stop}")
        if stop != "tool_use":
            return "".join(b.get("text", "") for b in out["content"]), turn
        sigs, results = [], []
        for block in out["content"]:
            if "toolUse" in block:
                tu = block["toolUse"]
                sigs.append((tu["name"], json.dumps(tu["input"], sort_keys=True)))
                res = TOOL_IMPL.get(tu["name"], lambda **k: {"error": "unknown tool"})(**tu["input"])
                if verbose: print(f"           -> {tu['name']}({tu['input']})")
                results.append({"toolResult": {"toolUseId": tu["toolUseId"], "content": [{"json": res}]}})
        if sigs and sigs == last_sigs:                       # Fix 2: no new information
            return "No new information to act on. Escalating to a human agent.", turn
        last_sigs = sigs
        messages.append({"role": "user", "content": results})
    return "Reached the step limit. Escalating to a human agent.", max_turns
print("run_agent_v2 ready (max_turns + spin-detection + handoff)")

In [ ]:
# The same prompt that looped 50+ times in notebook 2 — now it resolves cleanly.
answer, turns = run_agent_v2("Confirm and pay for rebooking option 1 on PNR ABC123 right now.")
print(f"\n--- resolved in {turns} turns ---\n{answer}")
# Expect: the model calls handoff_to_human (a legal move) and stops with end_turn. No runaway.

## 2. Inference parameters

`temperature` is the "how surprising can the next token be" dial. Same agent, different sub-tasks, different settings:

- **Extract a PNR** → `temperature = 0` (same answer every time)
- **Draft a customer apology** → `0.5–0.7` (some variation reads as human)

In [ ]:
extract = converse_text(CLAUDE_MODEL,
    "Extract just the PNR from: 'hi my booking ref is abc123 and im stuck'. Reply with only the PNR.",
    temperature=0.0, max_tokens=20)
print("extraction (temp 0):", extract)

apology = converse_text(CLAUDE_MODEL,
    "Write a one-sentence apology to a passenger whose flight was cancelled due to fog.",
    temperature=0.7, max_tokens=60)
print("apology   (temp 0.7):", apology)

### Dynamic control — vary it per call

Params do not drift on their own; **your code** changes them between turns. A legit pattern: run deterministic (temp 0); if the loop **stalls**, raise temperature **once** to break a repeated identical reply, then stop. Do not crank it blindly — high temperature on tool-argument generation produces malformed inputs and *more* loops.

In [ ]:
# pass a schedule that nudges temperature up each turn (illustrative)
schedule = lambda turn: min(0.0 + 0.25 * (turn - 1), 1.0)
answer, turns = run_agent_v2("PNR ABC123 — what happened and what are my options?",
                             temp_schedule=schedule, verbose=True)
print("\nanswer:", answer[:200])

### When NOT to touch them

- Deterministic extraction / routing → `temperature 0`; leave `topP` / `topK` alone.
- Change **one** dial, then measure. Tuning three at once teaches you nothing.
- Most agent bugs are **tool or prompt** bugs, not sampling bugs. Tuning temperature to fix a missing-tool loop is aspirin for a broken leg. Reach for params **last**.

## 2b. Reasoning models — thinking on = sampling off

Turn on extended thinking and **`temperature` / `topP` / `topK` become off-limits** (the call 400s). What you control instead is **how much** thinking:

- **Claude 3.7 / Claude 4** accept a manual `budget_tokens`.
- **Newest Opus (4.6 recommended; 4.7 / 4.8 required)** use **adaptive** thinking + an **`effort`** knob; manual `budget_tokens` is rejected on 4.7 / 4.8.

In [ ]:
# (a) manual budget on Claude 3.7 — note: do NOT also set temperature/topP/topK
try:
    resp = bedrock_runtime.converse(
        modelId=CLAUDE_37,
        messages=[{"role": "user", "content": [{"text": "A passenger missed a fog-cancelled flight. "
                                                        "List the 2 steps an agent should take. Be brief."}]}],
        additionalModelRequestFields={"thinking": {"type": "enabled", "budget_tokens": 1024}},
        inferenceConfig={"maxTokens": 800},   # maxTokens is fine; temperature is NOT
    )
    txt = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    print("3.7 + thinking OK:\n", txt[:400])
except botocore.exceptions.ClientError as e:
    print("3.7 thinking call failed ->", e.response["Error"]["Code"], e.response["Error"]["Message"][:140])

In [ ]:
# (b) prove the incompatibility: thinking ON + temperature set -> 400
try:
    bedrock_runtime.converse(
        modelId=CLAUDE_37,
        messages=[{"role": "user", "content": [{"text": "hi"}]}],
        additionalModelRequestFields={"thinking": {"type": "enabled", "budget_tokens": 1024}},
        inferenceConfig={"temperature": 0.2, "maxTokens": 800},   # <-- illegal with thinking on
    )
    print("unexpectedly succeeded")
except botocore.exceptions.ClientError as e:
    print("as expected ->", e.response["Error"]["Code"], ":", e.response["Error"]["Message"][:160])

In [ ]:
# (c) adaptive + effort on the newest Opus. Paste the CURRENT Opus profile id from the
#     console catalog (Bedrock > Model access). Ids change; do not hard-code an old one.
OPUS_MODEL = "us.anthropic.claude-opus-4-...-v1:0"   # <-- replace with the live id
try:
    resp = bedrock_runtime.converse(
        modelId=OPUS_MODEL,
        messages=[{"role": "user", "content": [{"text": "Summarize the agent loop in one line."}]}],
        additionalModelRequestFields={"thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}},
        inferenceConfig={"maxTokens": 400},   # no temperature/topP/topK; no budget_tokens on 4.7/4.8
    )
    print("Opus adaptive OK:", "".join(b.get("text", "") for b in resp["output"]["message"]["content"])[:300])
except botocore.exceptions.ClientError as e:
    print("Opus call ->", e.response["Error"]["Code"], "(paste the current Opus id to run this)")

## 3. Hallucination — defense in depth

A hallucination is an answer **not grounded** in the tool outputs (e.g. a flight number no tool returned). Five layers, cheap to expensive:

1. **Prompt** — "use only tool facts" (already in `SYSTEM`)
2. **Ground** — the only source the model sees is the tool output
3. **Guardrail** — contextual grounding check that **BLOCKs** low-grounding answers
4. **Custom parser** — an override Lambda that forces a reprompt (advanced)
5. **Verifier agent** — a second agent checks the first

Layers 1–2 are free and always on. Layer 3 is the Bedrock-native workhorse — we build it now.

### Layer 3 — create a contextual grounding guardrail

Two scores per response: **GROUNDING** (is the answer supported by the source?) and **RELEVANCE** (does it answer the query?). Each has a threshold (0–1) and an action: `BLOCK` or `NONE` (monitor only). Idempotent: we reuse an existing guardrail of the same name if present.

In [ ]:
GUARDRAIL_NAME = "travelmind-grounding"

def ensure_guardrail():
    # reuse if it already exists
    for g in bedrock.list_guardrails().get("guardrails", []):
        if g["name"] == GUARDRAIL_NAME:
            return g["id"]
    r = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Contextual grounding for TravelMind answers",
        contextualGroundingPolicyConfig={"filtersConfig": [
            {"type": "GROUNDING",  "threshold": 0.7, "action": "BLOCK"},
            {"type": "RELEVANCE",  "threshold": 0.7, "action": "BLOCK"},
        ]},
        blockedInputMessaging="I can only answer from your booking record.",
        blockedOutputsMessaging="I can only answer from your booking record.",
    )
    return r["guardrailId"]

GUARDRAIL_ID = ensure_guardrail()
print("guardrail id:", GUARDRAIL_ID)
# Tip: set action 'NONE' first to OBSERVE scores without blocking (you are not charged for a
# disabled filter), tune the threshold against real traffic, then flip to 'BLOCK'.

### Layer 3 in code — `apply_guardrail` on a good vs an invented answer

`apply_guardrail` scores a candidate answer **independently of generation** — works with any model, including the hand-built loop. Compare a grounded answer with one that invents a flight not in the source.

In [ ]:
SOURCE = "Booking ABC123 CANCELLED. Reason: fog at DEL. Options: 6E-415 18:40, 6E-422 21:10."
QUERY  = "My flight was cancelled. What can I rebook onto?"

def grounding_check(answer):
    resp = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID, guardrailVersion="DRAFT", source="OUTPUT",
        content=[
            {"text": {"text": SOURCE, "qualifiers": ["grounding_source"]}},
            {"text": {"text": QUERY,  "qualifiers": ["query"]}},
            {"text": {"text": answer}},
        ],
    )
    scores = {}
    for a in resp.get("assessments", []):
        for f in a.get("contextualGroundingPolicy", {}).get("filters", []):
            scores[f["type"]] = round(f.get("score", 0.0), 3)
    return resp["action"], scores

good = "You can rebook onto 6E-415 at 18:40 or 6E-422 at 21:10."
bad  = "You're confirmed on flight 6E-999 departing at 07:15 tomorrow."   # invented flight

print("GOOD answer ->", grounding_check(good))
print("BAD  answer ->", grounding_check(bad))
# Expect the invented answer to score low on GROUNDING and come back GUARDRAIL_INTERVENED.

In [ ]:
# wire it into the loop: draft, then refuse to send an ungrounded answer
def answer_with_grounding(user_text):
    draft, _ = run_agent_v2(user_text, verbose=False)
    action, scores = grounding_check(draft)
    if action == "GUARDRAIL_INTERVENED":
        return "Let me re-check your booking details before I answer.", scores
    return draft, scores

reply, scores = answer_with_grounding("PNR ABC123 — what are my options?")
print("scores:", scores, "\nreply:", reply[:200])

### Honest limits of the guardrail

- Built for **summarization / paraphrase / QA over a source** — not native multi-turn chat.
- Grounding score **degrades over long conversations**; pass the *relevant* tool output as the source each turn, not the whole transcript.
- Limits: source ≤ 100k chars, query ≤ 1k, response ≤ 5k.
- It **reduces** hallucination, it does **not** guarantee zero. Never promise a client "no hallucinations" — promise a measurable, tunable reduction with an audit trail.

### Layer 4 — the custom parser / override Lambda (advanced)

Replace Bedrock's built-in parser with **your** Lambda, per phase, to validate reasoning or force a reprompt. Powerful but high-maintenance: the **trace is produced by the parser**, so a broken parser blinds your main debugging tool. Use only for regulated / advanced cases. The config (do not run casually):

In [ ]:
# Illustration only — attaches a custom parser Lambda to the POST_PROCESSING phase.
def attach_custom_parser(agent_id, role_arn, model_id, parser_lambda_arn):
    ba = boto3.client("bedrock-agent", region_name=REGION)
    ba.update_agent(
        agentId=agent_id, agentName="TravelMind-UC1",
        agentResourceRoleArn=role_arn, foundationModel=model_id, instruction="...",
        promptOverrideConfiguration={
            "overrideLambda": parser_lambda_arn,
            "promptConfigurations": [{
                "promptType": "POST_PROCESSING",     # or ORCHESTRATION / PRE_PROCESSING / KNOWLEDGE_BASE_RESPONSE_GENERATION
                "parserMode": "OVERRIDDEN",          # use my Lambda for this phase
                "promptCreationMode": "DEFAULT",     # keep AWS's default prompt text
                "promptState": "ENABLED",
            }],
        },
    )
    ba.prepare_agent(agentId=agent_id)
print("attach_custom_parser defined (not executed). Prefer the verifier flow below for most cases.")

### Layer 5 — the verifier agent (preferred over nesting)

Instead of a parser that secretly calls another agent, run an **open flow**: Agent A drafts, Agent B checks the draft against the source, loop until clean. Slower (two model calls), but every step is visible and fixable. Here is a minimal verifier in code.

In [ ]:
def verify(answer, source, query, model_id=CLAUDE_MODEL):
    judge = (f"SOURCE FACTS:\n{source}\n\nUSER QUERY:\n{query}\n\nPROPOSED ANSWER:\n{answer}\n\n"
             "Is every factual claim in the proposed answer supported by SOURCE FACTS? "
             "Reply exactly 'PASS' if yes, otherwise 'FAIL: <what is unsupported>'.")
    return converse_text(model_id, judge, temperature=0.0, max_tokens=120).strip()

def drafted_and_verified(query, source, max_rounds=2):
    draft = converse_text(CLAUDE_MODEL,
        f"SOURCE FACTS:\n{source}\n\nAnswer the user using ONLY these facts.\nUser: {query}",
        system=SYSTEM[0]["text"], temperature=0.0, max_tokens=200)
    for _ in range(max_rounds):
        verdict = verify(draft, source, query)
        if verdict.startswith("PASS"):
            return draft, "PASS"
        # redraft with the verifier's objection
        draft = converse_text(CLAUDE_MODEL,
            f"SOURCE FACTS:\n{source}\n\nYour previous answer was rejected: {verdict}\n"
            f"Rewrite using ONLY the source facts.\nUser: {query}",
            system=SYSTEM[0]["text"], temperature=0.0, max_tokens=200)
    return draft, verdict

ans, status = drafted_and_verified("What can I rebook onto?", SOURCE)
print("status:", status, "\nanswer:", ans)

## Reach-for table

| When you hit… | Reach for |
|---|---|
| Runaway loop | `max_turns` + spin-detection + **fallback tool** |
| Need determinism | `temperature = 0` |
| Reasoning model | adaptive thinking + `effort` (no sampling params) |
| Ungrounded answers | grounding guardrail (`BLOCK`) + tool-only source |
| Regulated / custom output | override-Lambda parser (last resort) |
| Verify before answering | verifier-agent flow |

Order of reach: fix **tools and prompt first**, params second, guardrails for grounding, parser/verifier only when the stakes demand it.

**Next — Notebook 5:** production practices and industry insights — IAM roles, retries, observability, cost controls, evals, and security.